# Crypto time-series: what's predictable, what isn't

BTC 1-minute data (`data/defi.duckdb`, Coinbase, 14 days). The honest contrast:

- **Price direction is ~unpredictable** — efficient market, the move is already priced. Fourier shows a ~flat (white) return spectrum; the ACF is ~0 at every lag; a next-bar direction model lands at ~50%.
- **Volatility & volume ARE predictable** — volatility clusters (slow-decaying ACF, high forecast R²) and volume has a real **intraday cycle** (a 24h Fourier peak).

The craft takeaway: in an efficient market you forecast *risk and microstructure*, not direction. Fourier here is a **diagnostic**, not a prediction method.

In [ ]:
import sys
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.signal import periodogram
import os as _os
if _os.getenv('NB_DARK'):
    sns.set_theme(style='darkgrid', rc={
        'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','savefig.facecolor':'#0d1117',
        'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9','text.color':'#c9d1d9',
        'axes.titlecolor':'#c9d1d9','xtick.color':'#8b949e','ytick.color':'#8b949e',
        'grid.color':'#21262d'})
else:
    sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
while not (ROOT/'data').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
con = duckdb.connect(str(ROOT/'data'/'defi.duckdb'), read_only=True)
df = con.execute("SELECT t, close, volume FROM cex_candles WHERE venue='coinbase' AND asset='BTC' ORDER BY t").df()
con.close()
df['ret'] = np.log(df.close).diff()
df = df.dropna().reset_index(drop=True)
r = df.ret.to_numpy(); absr = np.abs(r)
print(f'{len(df)} 1-min BTC bars, {df.t.min()} -> {df.t.max()}')

## 1. Fourier spectra — returns are white, volume is seasonal

Power vs period. A flat line = no cyclical structure (white noise). A peak at 24h = a daily cycle.

In [ ]:
def spec(x, label, ax):
    x = x - np.mean(x)
    f, P = periodogram(x, fs=1.0)          # fs = 1 sample/min -> f in cycles/min
    m = f > 0; period_h = (1/f[m])/60.0
    ax.loglog(period_h, P[m], lw=.7); ax.set_title(label); ax.set_xlabel('period (hours)'); ax.axvline(24, color='crimson', lw=.8, ls='--')
fig, ax = plt.subplots(1,3, figsize=(15,4))
spec(r, 'returns (≈ flat = white)', ax[0])
spec(absr, '|returns| (volatility)', ax[1])
spec(df.volume.to_numpy(), 'volume (24h peak)', ax[2])
ax[0].set_ylabel('power'); plt.tight_layout(); plt.show()

## 2. Intraday seasonality (by hour of day, UTC)

The cycle Fourier hinted at, made explicit — volume and volatility both ebb and flow with the trading day.

In [ ]:
df['hour'] = df.t.dt.hour
prof = df.groupby('hour').agg(volume=('volume','mean'), volatility=('ret', lambda s: s.abs().mean()))
fig, ax = plt.subplots(1,2, figsize=(13,4))
ax[0].plot(prof.index, prof.volume, marker='o', color='steelblue'); ax[0].set_title('avg volume by hour (UTC)'); ax[0].set_xlabel('hour')
ax[1].plot(prof.index, prof.volatility*1e4, marker='o', color='darkorange'); ax[1].set_title('avg |return| by hour (bps)'); ax[1].set_xlabel('hour')
plt.tight_layout(); plt.show()

## 3. Autocorrelation — the stylized fact

Returns: ACF ≈ 0 at all lags (unpredictable). |returns|: slow decay (volatility clusters — today's calm/storm persists).

In [ ]:
def acf(x, L):
    x = x - x.mean(); v = x @ x
    return [1.0] + [float((x[:-k] @ x[k:]) / v) for k in range(1, L+1)]
L=120; lags=range(0,L+1)
plt.figure(figsize=(11,4))
plt.plot(list(lags), acf(r,L), label='returns', color='steelblue')
plt.plot(list(lags), acf(absr,L), label='|returns| (volatility)', color='darkorange')
plt.axhline(0,color='gray',lw=.6); plt.xlabel('lag (minutes)'); plt.ylabel('autocorrelation'); plt.legend(); plt.title('ACF: returns vs volatility'); plt.show()

## 4. Head-to-head: predicting direction vs volatility

Out-of-sample (chronological 80/20). Direction = a logistic on recent returns; volatility = an AR(3) on hourly realized variance. The R² gap is the whole point.

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score
def oos_r2(yhat, ytrue, ytrain_mean):
    ss_res=np.sum((ytrue-yhat)**2); ss_tot=np.sum((ytrue-ytrain_mean)**2); return 1-ss_res/ss_tot
# --- direction at 1-min: sign(r_{t+1}) from last 5 returns ---
K=5; Xd=np.column_stack([r[i:len(r)-K+i] for i in range(K)]); yd=(r[K:]>0).astype(int)
c=int(0.8*len(yd)); clf=LogisticRegression(max_iter=500).fit(Xd[:c],yd[:c])
dir_acc=accuracy_score(yd[c:], clf.predict(Xd[c:]))
# --- volatility: hourly realized variance, AR(3) ---
hourly=df.set_index('t').ret.pow(2).resample('1h').sum().dropna().to_numpy()
P=3; Xv=np.column_stack([hourly[i:len(hourly)-P+i] for i in range(P)]); yv=hourly[P:]
cv=int(0.8*len(yv)); reg=LinearRegression().fit(Xv[:cv],yv[:cv])
vol_r2=oos_r2(reg.predict(Xv[cv:]), yv[cv:], yv[:cv].mean())
# return-level R² for contrast
Xr=Xd.astype(float); yr=r[K:]; rr=LinearRegression().fit(Xr[:c],yr[:c]); ret_r2=oos_r2(rr.predict(Xr[c:]), yr[c:], yr[:c].mean())
print(f'next-min DIRECTION accuracy : {dir_acc:.4f}  (coin flip = 0.50)')
print(f'next-min RETURN  OOS R^2    : {ret_r2:+.4f}  (~0 = unpredictable)')
print(f'next-hour VOLATILITY OOS R^2: {vol_r2:+.4f}  (>0 = real, exploitable structure)')

## Verdict

The spectra, the ACF, and the head-to-head all say the same thing: **BTC return *direction* is ~unpredictable** (white spectrum, zero ACF, ~50% accuracy, ~0 R²) — the efficient-market reality, and the wrong target for the prediction craft. But **volatility and volume carry strong, exploitable structure** (clustering + a clean intraday cycle, high forecast R²).

So on the DeFi side the edge isn't 'predict the price' — it's **volatility/risk forecasting, intraday seasonality, funding mean-reversion, and CEX↔DEX lead-lag** (notebook 01). Same honest lesson as the sports market work: model what carries signal, not what's already priced.